In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

# Load Model & Scaled Test Matrix
rf_model = joblib.load('../models/Fraud_Data_random_forest.pkl')
X_test = np.load('../data/processed/X_test_f.npy')
y_test = np.load('../data/processed/y_test_f.npy')

# Re-establish placeholder names (Numerical + Cat dummies)
num_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup', 'device_velocity', 'ip_velocity']
feature_names = num_cols + [f"cat_feature_{i}" for i in range(X_test.shape[1] - len(num_cols))]

# ==========================================
# 1. Built-In Feature Importance (Top 10)
# ==========================================
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 4))
sns.barplot(x=importances[indices[:10]], y=[feature_names[i] for i in indices[:10]], palette='viridis')
plt.title('Top 10 Features (Built-In MDI)')
plt.show()

# ==========================================
# 2. SHAP Analysis
# ==========================================
# Background sample calculation (speeds up processing)
shap.initjs()
explainer = shap.TreeExplainer(rf_model)

# Sample to isolate TP, FP, FN cases from prediction array
preds = rf_model.predict(X_test)
tp_idx = np.where((preds == 1) & (y_test == 1))[0][0]
fp_idx = np.where((preds == 1) & (y_test == 0))[0][0]
fn_idx = np.where((preds == 0) & (y_test == 1))[0][0]

# Sample subset of test partition to compute collective values
sample_idx = np.concatenate([np.random.choice(len(X_test), 200, replace=False), [tp_idx, fp_idx, fn_idx]])
X_sample = X_test[sample_idx]

shap_values = explainer.shap_values(X_sample)
shap_val_class1 = shap_values[1] if isinstance(shap_values, list) else shap_values

# Generate SHAP Summary Plot
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_val_class1, X_sample, feature_names=feature_names, show=False)
plt.title("SHAP Global Summary Plot", fontsize=14)
plt.tight_layout()
plt.show()

# Helper for local force plots
expected_value = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value

def display_force_plot(idx, label):
    # Compute local row values
    row_shap = explainer.shap_values(X_test[idx:idx+1])
    row_shap_class1 = row_shap[1] if isinstance(row_shap, list) else row_shap
    
    plt.figure(figsize=(12, 3))
    shap.force_plot(
        expected_value, 
        row_shap_class1[0], 
        X_test[idx], 
        feature_names=feature_names, 
        matplotlib=True, 
        show=False
    )
    plt.title(f"SHAP Force Plot: {label} (Index {idx})", pad=20)
    plt.tight_layout()
    plt.show()

# Generate the three specified Force Plots
display_force_plot(tp_idx, "True Positive (Correctly Classified Fraud)")
display_force_plot(fp_idx, "False Positive (Legitimate Client Flagged as Fraud)")
display_force_plot(fn_idx, "False Negative (Uncaught Fraud Event)")